In [ ]:
문제1 코드

In [ ]:
import csv
import json
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(message)s')

def make_report(csv_path: str, json_path: str) -> int:
    students_data = []
    success_count = 0
    
    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                name = row["이름"].strip()
                student_id = row["학번"].strip()
            
                try:
                    scores = {
                        "중간": int(row["중간"]) if row["중간"] else None,
                        "기말": int(row["기말"]) if row["기말"] else None,
                        "과제": int(row["과제"]) if row["과제"] else None
                    }
                except ValueError:
                    scores = {"중간": None, "기말": None, "과제": None}

                
                if None in scores.values():
                    avg, grade = None, None
                    logging.info(f"{name}: 결측값이 있습니다.")
                else:
                    avg = (scores["중간"] * 0.3) + (scores["기말"] * 0.5) + (scores["과제"] * 0.2)
                    
                    if avg >= 90: grade = "A"
                    elif avg >= 80: grade = "B"
                    elif avg >= 70: grade = "C"
                    else: grade = "F"
                    logging.info(f"{name}: 평균 {avg:.1f}, 등급 {grade}")
                    success_count += 1

                students_data.append({
                    "이름": name,
                    "학번": student_id,
                    "점수": scores,
                    "평균": avg,
                    "등급": grade
                })
                
    except FileNotFoundError:
        logging.warning(f"파일이 존재하지 않습니다: {csv_path}")
        return 0
    except UnicodeDecodeError:
        logging.error(f"인코딩 오류가 발생했습니다: {csv_path}")
        return 0
    
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(students_data, f, ensure_ascii=False, indent=4)
        
    return success_count

In [ ]:
#결과
INFO:김언어: 평균 89.5, 등급 B
INFO:이국문: 평균 84.4, 등급 B
INFO:박영문: 평균 93.5, 등급 A
INFO:최역사: 결측값이 있습니다.

In [ ]:
#json
[
    {
        "이름": "김언어",
        "학번": "2026-10000",
        "점수": {"중간": 85, "기말": 92, "과제": 90},
        "평균": 89.5,
        "등급": "B"
    },
    {
        "이름": "이국문",
        "학번": "2026-12345",
        "점수": {"중간": 78, "기말": 88, "과제": 85},
        "평균": 84.4,
        "등급": "B"
    },
    {
        "이름": "박영문",
        "학번": "2026-13579",
        "점수": {"중간": 95, "기말": 90, "과제": 100},
        "평균": 93.5,
        "등급": "A"
    },
    {
        "이름": "최역사",
        "학번": "2025-11111",
        "점수": {"중간": null, "기말": 82, "과제": 88},
        "평균": null,
        "등급": null
    }
]

설명:csv.DictReader를 사용해 각 행을 학생별 딕셔너리로 읽었고 점수 칸에 빈 문자열이 하나라도 있으면 평균과 등급을 계산하지 않고 None으로 처리하여 json에서는 null로 저장되게 했다. 한글 데이터이므로 파일을 열 때 encoding="utf-8"를 사용하였고 파일 없음과 인코딩 오류는 각각 warning,error 로그로 처리했다.

In [5]:
#q2
class InvalidJamoError(ValueError):
    pass


def classify_jamo(c: str) -> str:
    if not isinstance(c, str):
        raise TypeError(f"문자열이 아닙니다: {c}")

    if len(c) != 1:
        raise ValueError(f"길이가 1이 아닙니다: {c}")

    code: int = ord(c)

    if 0x3131 <= code <= 0x314E:
        return "자음"
    elif 0x314F <= code <= 0x3163:
        return "모음"
    else:
        raise InvalidJamoError(f"한글 자모가 아닙니다: {c}")

In [6]:
#실행코드
inputs = ["ㄱ", "ㅏ", "ㄲ", "가", "AB", 5, "ㅎ", "ㅣ", ""]

for item in inputs:
    try:
        result = classify_jamo(item)
        print(result)
    except TypeError as e:
        print(f"[TypeError] {e}")
    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")
    except ValueError as e:
        print(f"[ValueError] {e}")

자음
모음
자음
[InvalidJamoError] 한글 자모가 아닙니다: 가
[ValueError] 길이가 1이 아닙니다: AB
[TypeError] 문자열이 아닙니다: 5
자음
모음
[ValueError] 길이가 1이 아닙니다: 


InvalidJamoError는 입력값의 타입이나 길이 문제가 아니라 값은 문자열 한 글자이지만 허용된 자모 범위가 아니라는 의미이므로 Exception보다 ValueError의 자식으로 만드는 것이 적절하다. InvalidJamoError가 ValueError의 자식이기 때문에 except InvalidJamoError를 except ValueError보다 먼저 작성해야한다. 그렇지 않으면 InvalidJamoError도 일반 ValueError로 먼저 걸리게 된다